# Test BayesCard

This notebook connects BayesCard (pure BN path) to the existing experiment system.

- Reuses `experiment/BayesCardRunner.py` for training and inference orchestration
- Outputs to `experiment/checkpoint/BayesCard/`
- First evaluates feasibility for four benchmarks, then executes only supported ones

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = (Path.cwd().parent if (Path.cwd().parent / 'Benchmark').exists() else Path.cwd()).resolve()
EXPERIMENT_DIR = PROJECT_ROOT / 'experiment'
CHECKPOINT_DIR = EXPERIMENT_DIR / 'checkpoint' / 'BayesCard'

sys.path.insert(0, str(EXPERIMENT_DIR))
from BayesCardRunner import BayesCardRunner

runner = BayesCardRunner(PROJECT_ROOT)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f'PROJECT_ROOT={PROJECT_ROOT}')
print(f'CHECKPOINT_DIR={CHECKPOINT_DIR}')

/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.4) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/site-packages/pgmpy/utils/utils.py:4: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soo

PROJECT_ROOT=/home/liwei/starce-final/StarCE
CHECKPOINT_DIR=/home/liwei/starce-final/StarCE/experiment/checkpoint/BayesCard


In [2]:
ALL_BENCHMARKS = ['STATS', 'JOBLight', 'JOBLightRanges', 'JOBM', 'StatsJoin']
SEED = 0

capability_rows = []
for benchmark in ALL_BENCHMARKS:
    supported, reason = runner.evaluate_benchmark_capability(benchmark)
    capability_rows.append(
        {
            'benchmark': benchmark,
            'supported_by_pure_bayescard_bn': supported,
            'reason': reason,
        }
    )

capability_df = pd.DataFrame(capability_rows)
capability_df

,benchmark,supported_by_pure_bayescard_bn,reason
0,STATS,True,Supported: can follow the pure BayesCard train...
1,JOBLight,True,Supported: can follow the pure BayesCard train...
2,JOBLightRanges,True,Supported: can follow the pure BayesCard train...
3,JOBM,False,Does not support pure BayesCard: JOBM's reliab...
4,StatsJoin,True,"Supported: reuses STATS BN model, inference on..."


## Execute

Only execute supported benchmarks below (`STATS`, `JOBLight`, `JOBLightRanges`, `StatsJoin`).

`JOBM` in the current repo relies on sampling for the reliable path, not pure BayesCard/BN, this notebook does not execute it.
`StatsJoin` reuses STATS BN model, inference only, no training.

In [3]:
BENCHMARKS_TO_RUN = [
    row['benchmark']
    for row in capability_rows
    if row['supported_by_pure_bayescard_bn']
]

summary_df = runner.run_all(BENCHMARKS_TO_RUN, seed=SEED, force_retrain=True)
summary_df

INFO:DataPrepare.prepare_single_tables:Preparing hdf file for table badges
INFO:DataPrepare.prepare_single_tables:Managing functional dependencies for table badges
INFO:DataPrepare.prepare_single_tables:Preparing multipliers for table badges
INFO:DataPrepare.prepare_single_tables:Preparing categorical values and null values for table badges
INFO:DataPrepare.prepare_single_tables:Relevant attributes for table badges are ['Id', 'UserId', 'Date']
INFO:DataPrepare.prepare_single_tables:NULL values for table badges are [-145430.05460150907, -118638.27789238832, -83944821.66928385]


!!!!!!!!!!!!!!!!!!!!!!!!!
Index(['badges.Id', 'badges.UserId', 'badges.Date'], dtype='object')
@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
[] Index(['badges.Id', 'badges.UserId', 'badges.Date'], dtype='object')


INFO:DataPrepare.prepare_single_tables:Adding table parts without join partners for table badges
INFO:DataPrepare.prepare_single_tables:Preparing hdf file for table votes
INFO:DataPrepare.prepare_single_tables:Managing functional dependencies for table votes
INFO:DataPrepare.prepare_single_tables:Preparing multipliers for table votes
INFO:DataPrepare.prepare_single_tables:Preparing categorical values and null values for table votes
INFO:DataPrepare.prepare_single_tables:Relevant attributes for table votes are ['Id', 'PostId', 'VoteTypeId', 'CreationDate', 'UserId', 'BountyAmount']
INFO:DataPrepare.prepare_single_tables:NULL values for table votes are [-291496.65403642703, -140805.12052162504, -100002.72127330764, -76489038.65597203, -114338.12160392365, -100067.34528348624]
INFO:DataPrepare.prepare_single_tables:Adding table parts without join partners for table votes
INFO:DataPrepare.prepare_single_tables:Preparing hdf file for table postHistory


!!!!!!!!!!!!!!!!!!!!!!!!!
Index(['votes.Id', 'votes.PostId', 'votes.VoteTypeId', 'votes.CreationDate',
       'votes.UserId', 'votes.BountyAmount'],
      dtype='object')
@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
[] Index(['votes.Id', 'votes.PostId', 'votes.VoteTypeId', 'votes.CreationDate',
       'votes.UserId', 'votes.BountyAmount'],
      dtype='object')


INFO:DataPrepare.prepare_single_tables:Managing functional dependencies for table postHistory
INFO:DataPrepare.prepare_single_tables:Preparing multipliers for table postHistory
INFO:DataPrepare.prepare_single_tables:Preparing categorical values and null values for table postHistory
INFO:DataPrepare.prepare_single_tables:Relevant attributes for table postHistory are ['Id', 'PostHistoryTypeId', 'PostId', 'CreationDate', 'UserId']
INFO:DataPrepare.prepare_single_tables:NULL values for table postHistory are [-288489.2569744702, -100004.90864489143, -156538.34982805562, -82263706.33163796, -116035.05473015195]
INFO:DataPrepare.prepare_single_tables:Adding table parts without join partners for table postHistory
INFO:DataPrepare.prepare_single_tables:Preparing hdf file for table posts
INFO:DataPrepare.prepare_single_tables:Managing functional dependencies for table posts
INFO:DataPrepare.prepare_single_tables:Preparing multipliers for table posts
INFO:DataPrepare.prepare_single_tables:Prepari

...

Structure learning took 8.456488609313965 secs.


INFO:Models.Bayescard_BN:done, took 3.63533616065979 secs.
INFO:test_benchmark:Loaded BN model: 0_chow-liu_1.pkl
INFO:test_benchmark:Loaded BN model: 1_chow-liu_1.pkl
INFO:test_benchmark:Loaded BN model: 2_chow-liu_1.pkl
INFO:test_benchmark:Loaded BN model: 3_chow-liu_1.pkl
INFO:test_benchmark:Loaded BN model: 4_chow-liu_1.pkl
INFO:test_benchmark:Loaded 5 BN models from /home/liwei/starce-final/StarCE/experiment/checkpoint/BayesCard/models/joblightranges


done, parameter learning took 3.636654853820801 secs.


INFO:test_benchmark:Loaded BN model: 0_chow-liu_1.pkl
INFO:test_benchmark:Loaded BN model: 10_chow-liu_1.pkl
INFO:test_benchmark:Loaded BN model: 1_chow-liu_1.pkl
INFO:test_benchmark:Loaded BN model: 2_chow-liu_1.pkl
INFO:test_benchmark:Loaded BN model: 3_chow-liu_1.pkl
INFO:test_benchmark:Loaded BN model: 4_chow-liu_1.pkl
INFO:test_benchmark:Loaded BN model: 5_chow-liu_1.pkl
INFO:test_benchmark:Loaded BN model: 6_chow-liu_1.pkl
INFO:test_benchmark:Loaded BN model: 7_chow-liu_1.pkl
INFO:test_benchmark:Loaded BN model: 8_chow-liu_1.pkl
INFO:test_benchmark:Loaded BN model: 9_chow-liu_1.pkl
INFO:test_benchmark:Loaded 11 BN models from /home/liwei/starce-final/StarCE/experiment/checkpoint/BayesCard/models/stats


,benchmark,seed,preprocess_time_sec,train_time_sec,total_build_like_time_sec,total_eval_like_time_sec,query_count,avg_latency_sec,checkpoint_output,model_dir,statistics_size_bytes
0,STATS,0,28.03348,114.880966,142.914446,40.207078,2471,0.013303,/home/liwei/starce-final/StarCE/experiment/che...,/home/liwei/starce-final/StarCE/experiment/che...,6326548
1,JOBLight,0,0.00000,724.880644,724.880644,3.081380,451,0.004361,/home/liwei/starce-final/StarCE/experiment/che...,/home/liwei/starce-final/StarCE/experiment/che...,1630079
2,JOBLightRanges,0,33.53912,1014.735962,1048.275082,70.893444,8292,0.006208,/home/liwei/starce-final/StarCE/experiment/che...,/home/liwei/starce-final/StarCE/experiment/che...,2309697
3,STATSJOIN,0,0.00000,0.000000,0.000000,2.455658,226,0.008057,/home/liwei/starce-final/StarCE/experiment/che...,/home/liwei/starce-final/StarCE/experiment/che...,6326548


In [4]:
summary_csv = CHECKPOINT_DIR / 'benchmark_times.csv'
summary_df = pd.read_csv(summary_csv)

print(f'Summary CSV: {summary_csv}')
for _, row in summary_df.iterrows():
    print(f"{row['benchmark']}: card={row['checkpoint_output']}, eval_time={row['total_eval_like_time_sec']}")

summary_df

Summary CSV: /home/liwei/starce-final/StarCE/experiment/checkpoint/BayesCard/benchmark_times.csv
STATS: card=/home/liwei/starce-final/StarCE/experiment/checkpoint/BayesCard/card_stats.txt, eval_time=40.20707845687866
JOBLight: card=/home/liwei/starce-final/StarCE/experiment/checkpoint/BayesCard/card_joblight.txt, eval_time=3.0813801288604736
JOBLightRanges: card=/home/liwei/starce-final/StarCE/experiment/checkpoint/BayesCard/card_joblr.txt, eval_time=70.89344382286072
STATSJOIN: card=/home/liwei/starce-final/StarCE/experiment/checkpoint/BayesCard/card_statsjoin.txt, eval_time=2.455657958984375


,benchmark,total_eval_like_time_sec,checkpoint_output,train_time_sec,total_build_like_time_sec,preprocess_time_sec,query_count,avg_latency_sec,model_dir,statistics_size_bytes,seed
0,STATS,40.207078,/home/liwei/starce-final/StarCE/experiment/che...,114.880966,142.914446,28.03348,2471,0.013303,/home/liwei/starce-final/StarCE/experiment/che...,6326548,0
1,JOBLight,3.081380,/home/liwei/starce-final/StarCE/experiment/che...,724.880644,724.880644,0.00000,451,0.004361,/home/liwei/starce-final/StarCE/experiment/che...,1630079,0
2,JOBLightRanges,70.893444,/home/liwei/starce-final/StarCE/experiment/che...,1014.735962,1048.275082,33.53912,8292,0.006208,/home/liwei/starce-final/StarCE/experiment/che...,2309697,0
3,STATSJOIN,2.455658,/home/liwei/starce-final/StarCE/experiment/che...,0.000000,0.000000,0.00000,226,0.008057,/home/liwei/starce-final/StarCE/experiment/che...,6326548,0
